# Projet Data Mining – Segmentation RFM par K‑means

*Etudiant :* Aboubekrine Abderrahmane ouerzeg
*Matricules :*  C29371 
*Objectif :* Regrouper les clients selon leur comportement d’achat (Récence, Fréquence, Montant) pour proposer des actions marketing ciblées.

1. Imports et configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from mpl_toolkits.mplot3d import Axes3D

%matplotlib inline
sns.set_style('whitegrid')

2. Chargement du dataset

In [ ]:
df = pd.read_csv('../data/Online Retail Dataset.csv', encoding='unicode_escape')
print("Forme du DataFrame brut :", df.shape)
df.head()

In [ ]:
print(df.columns.tolist())

3. Nettoyage des données

· Suppression des lignes sans Customer ID
· Suppression des quantités ≤ 0 (annulations)
· Suppression des prix ≤ 0
· Création de la colonne TotalAmount = Quantity × Price

In [ ]:
# Supprimer les lignes sans identifiant client
df_clean = df.dropna(subset=['CustomerID'])

# Quantités strictement positives
df_clean = df_clean[df_clean['Quantity'] > 0]

# Prix strictement positifs
df_clean = df_clean[df_clean['UnitPrice'] > 0]

# Montant total par ligne
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

print("Forme après nettoyage :", df_clean.shape)

4. Feature Engineering – Calcul des métriques RFM

La date de référence est la date maximale des transactions + 1 jour.
On agrège par Customer ID :

· Récence : (date de référence – date de dernière commande) en jours
· Fréquence : nombre de factures distinctes (Invoice)
· Montant : somme de TotalAmount

In [ ]:
# Conversion en datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'],dayfirst=True)
reference_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)
print("Date de référence :", reference_date)

# Agrégation
rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,
    'InvoiceNo': lambda x: x.nunique(),
    'TotalAmount': 'sum'
}).rename(columns={
    'InvoiceDate': 'Recence',
    'InvoiceNo': 'Frequence',
    'TotalAmount': 'Montant'
})

rfm.head()

5. Transformation et standardisation

Le K‑means est sensible aux échelles et aux valeurs extrêmes.

· Transformation logarithmique log1p sur Frequence et Montant
· Standardisation (moyenne = 0, écart‑type = 1) des trois variables : Recence, Frequence_log, Montant_log

In [ ]:
from sklearn.preprocessing import	StandardScaler
rfm['Frequence_log'] = np.log1p(rfm['Frequence'])
rfm['Montant_log'] = np.log1p(rfm['Montant'])

features = ['Recence', 'Frequence_log', 'Montant_log']
rfm_scaled = StandardScaler().fit_transform(rfm[features])

print("Moyennes après standardisation :", rfm_scaled.mean(axis=0).round(2))
print("Écarts-types :", rfm_scaled.std(axis=0).round(2))

6. Choix du nombre de clusters (k)

On utilise la méthode du coude (inertie) et le silhouette score pour k de 2 à 10.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
inertias = []
sil_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(rfm_scaled)
    inertias.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, labels))

# Visualisation
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('k')
plt.ylabel('Inertie')
plt.title('Méthode du coude')

plt.subplot(1,2,2)
plt.plot(K_range, sil_scores, 'ro-')
plt.xlabel('k')
plt.ylabel('Silhouette score')
plt.title('Score de silhouette')
plt.tight_layout()
plt.show()

# Choisir k (exemple : 4)
k_optimal = 4
print(f"k optimal retenu : {k_optimal}")

7. Clustering final

In [ ]:
kmeans_final = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
rfm['Cluster'] = kmeans_final.fit_predict(rfm_scaled)

8. Analyse des clusters

8.1 Profil moyen (valeurs brutes)

In [ ]:
cluster_profile = rfm.groupby('Cluster').agg({
    'Recence': 'mean',
    'Frequence': 'mean',
    'Montant': 'mean'
}).round(2)

print(cluster_profile)

plt.figure(figsize=(8,5))
sns.heatmap(cluster_profile, annot=True, cmap='coolwarm', fmt='.0f')
plt.title('Valeurs moyennes par cluster (échelles originales)')
plt.show()

8.2 Visualisations 2D

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

sns.scatterplot(data=rfm, x='Frequency_log', y='Monetary_log', hue='Cluster', palette='Set1', ax=axes[0])
axes[0].set_title('Fréquence (log) vs Montant (log)')

sns.scatterplot(data=rfm, x='Recency', y='Monetary_log', hue='Cluster', palette='Set1', ax=axes[1])
axes[1].set_title('Récence vs Montant (log)')

plt.tight_layout()
plt.show()

8.3 Visualisation 3D

In [ ]:
fig = plt.figure(figsize=(12,8))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(rfm['Recency'], rfm['Frequency_log'], rfm['Monetary_log'],
                c=rfm['Cluster'], cmap='viridis', s=30, alpha=0.7)
ax.set_xlabel('Récence (jours)')
ax.set_ylabel('Fréquence (log)')
ax.set_zlabel('Montant (log)')
ax.set_title('Clusters RFM  Vue 3D')
plt.colorbar(sc, label='Cluster')
plt.show()

9. Interprétation métier et recommandations

Affichage détaillé de chaque cluster :

In [ ]:
for cluster in sorted(rfm['Cluster'].unique()):
    data = rfm[rfm['Cluster'] == cluster]
    print(f"\n--- Cluster {cluster} ({len(data)} clients) ---")
    print(f"  Récence moyenne : {data['Recence'].mean():.1f} jours")
    print(f"  Fréquence moyenne : {data['Frequence'].mean():.1f} commandes")
    print(f"  Dépense moyenne : {data['Montant'].mean():.2f} £")